# Phase 157: CryptoBench Ingestion + Topological Variability

Extends Phase42's exact ingestion (same fetch/parse/featurizer/dihedral/
ESM cells) with a new feature block: the per-residue STD of the
persistent-homology feature vector across an 8-conformer ANM-perturbed
ensemble, on top of the existing STATIC (single apo-structure) topology
block. Hypothesis: a cryptic pocket is, by definition, a cavity that is
not present in the static apo structure but exists in some nearby,
biologically plausible conformation -- so the *variability* of local
topology across plausible small conformational changes should carry
signal the static snapshot cannot. Zero architecture change: the
fusion module (`Fus`) just sees a wider topology block (34 static + 34
variability columns instead of 34).

## Cell 1: Install dependencies (identical to Phase42)

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3

## Cell 2: Setup, fetch CryptoBench's dataset.json + folds.json from OSF (identical to Phase42)

In [ ]:
import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import gudhi
from scipy.spatial import cKDTree
from torch_geometric.data import Data
from Bio.PDB import MMCIFParser, PDBParser, PPBuilder

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

def fetch_json(url):
    with urllib.request.urlopen(url, timeout=120) as r:
        return json.loads(r.read().decode())

cb_dataset = fetch_json("https://osf.io/download/ta2ju/")
cb_folds = fetch_json("https://osf.io/download/5s93p/")
print(f"CryptoBench: {len(cb_dataset):,} apo entries | folds: "
      f"{ {k: len(v) for k,v in cb_folds.items()} }")

# --- pick the MAIN holo pairing (unique per apo entry, verified: exactly
# one is_main_holo_structure==True per apo_id, 0 zero-main / 0 multi-main) --
units = []
for apo_id, entries in cb_dataset.items():
    main = next((e for e in entries if e.get("is_main_holo_structure")), None)
    if main is None: continue
    # keep as string, not int: residue numbers can carry a PDB insertion
    # code suffix (e.g. "60A"), which int() rejects
    pocket_resnums = {sel.split("_", 1)[1] for sel in main["apo_pocket_selection"]}
    units.append({"apo_id": apo_id, "apo_chain": main["apo_chain"],
                   "pocket_resnums": pocket_resnums, "pRMSD": main.get("pRMSD")})
print(f"training units (one main pocket per apo entry): {len(units):,}")

split_of = {}
for pid in cb_folds.get("test", []): split_of[pid] = "test"
for fold_key in ("train-0","train-1","train-2","train-3"):
    for pid in cb_folds.get(fold_key, []): split_of[pid] = "train"
for u in units:
    u["split"] = split_of.get(u["apo_id"], "unassigned")
n_unassigned = sum(1 for u in units if u["split"] == "unassigned")
print(f"split coverage: {sum(1 for u in units if u['split']!='unassigned')}/{len(units)} "
      f"({n_unassigned} unassigned, dropped)")
units = [u for u in units if u["split"] != "unassigned"]

## Cell 3: Fetch apo structures (identical to Phase42)

In [ ]:
need = sorted({u["apo_id"] for u in units})
def fetch(pid):
    for ext in ("cif","pdb"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("cif","pdb"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 4: Parse apo chains, build per-residue crypticity labels (identical to Phase42)

In [ ]:
AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
MIN_RES, MAX_RES = 30, 1500

def parse_unit(u):
    pid, cid = u["apo_id"], u["apo_chain"]
    if pid not in paths: return None
    path = paths[pid]
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path, "rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[cid]
    except Exception:
        return None
    residues = [r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if not (MIN_RES <= len(residues) <= MAX_RES): return None
    seq, ca, per_res, resnums = [], [], [], []
    for r in residues:
        seq.append(AA3_TO_1.get(r.get_resname().strip().upper(), "X"))
        ca.append(r["CA"].coord)
        atoms = [a.coord for a in r.get_unpacked_list() if a.element not in ("H","D")]
        per_res.append(np.asarray(atoms, dtype=np.float32).reshape(-1,3))
        resnums.append(str(r.id[1]) + r.id[2].strip())   # e.g. "60" or "60A" (insertion code)
    y = np.array([1.0 if rn in u["pocket_resnums"] else 0.0 for rn in resnums], dtype=np.float32)
    n_found = int(y.sum())
    if n_found < 3: return None   # too few pocket residues actually present in this chain -- skip
    return dict(pdb_id=pid, chain_id=cid, seq="".join(seq),
                ca=np.asarray(ca, dtype=np.float32), per_res=per_res, y=y,
                split=u["split"], pRMSD=u["pRMSD"], n_pocket_labeled=len(u["pocket_resnums"]),
                n_pocket_found=n_found)

from joblib import Parallel, delayed
t0 = time.time()
parsed = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(parse_unit)(u) for u in units)
records = [r for r in parsed if r is not None]
print(f"\nparsed in {(time.time()-t0)/60:.1f} min")
print(f"usable chains: {len(records):,}/{len(units):,}")
n_res = sum(len(r["y"]) for r in records)
n_pos = sum(int(r["y"].sum()) for r in records)
print(f"residues {n_res:,} | cryptic-pocket residues {n_pos:,} ({100*n_pos/max(n_res,1):.2f}%)")
for s in ("train","test"):
    rs = [r for r in records if r["split"] == s]
    print(f"  {s:<6} {len(rs):>5} chains")

## Cell 5: Featuriser (identical to Phase42 -- byte-for-byte the Phase 2b/6b/29/36 definition)

In [ ]:
MAX_HOM_DIM = 2
BETTI_RADII = (2.0, 3.0, 5.0)
NBHD_RADIUS = 12.0
MAX_NBHD_ATOMS = 300
FEATS_PER_DIM = 6 + 2 + len(BETTI_RADII)
TOPO_DIM = FEATS_PER_DIM * (MAX_HOM_DIM + 1) + 2

TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM + 1)
              for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                        "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]] \
             + ["nbhd_n_atoms", "nbhd_gyration"]
BURIAL_NAMES = ["nbhd_n_atoms", "nbhd_gyration"]


def _diagrams(pts):
    pts = np.ascontiguousarray(pts, dtype=np.float64)
    st, squared = None, True
    if len(pts) >= 5:
        try:
            st = gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception:
            st = None
    if st is None:
        st = gudhi.RipsComplex(points=pts, max_edge_length=float(NBHD_RADIUS)) \
                  .create_simplex_tree(max_dimension=MAX_HOM_DIM + 1)
        squared = False
    st.compute_persistence(homology_coeff_field=2, persistence_dim_max=True)
    out = {}
    for d in range(MAX_HOM_DIM + 1):
        b = np.asarray(st.persistence_intervals_in_dimension(d), dtype=np.float64).reshape(-1, 2)
        out[d] = np.sqrt(np.maximum(b, 0.0)) if (squared and len(b)) else b   # alpha -> angstrom
    return out


def _vectorize(dgms, pts):
    feats = []
    for d in range(MAX_HOM_DIM + 1):
        bars = dgms.get(d, np.zeros((0, 2)))
        fin = bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1, 2)
        if len(fin) == 0:
            feats.extend([0.0] * FEATS_PER_DIM); continue
        b, dth = fin[:, 0], fin[:, 1]
        pers = np.maximum(dth - b, 0.0); tot = float(pers.sum())
        if tot > 0:
            p = pers / tot; nz = p[p > 0]; ent = float(-(nz * np.log(nz)).sum())
        else:
            ent = 0.0
        feats.extend([float(len(fin)), tot, float(pers.max()), float(pers.mean()),
                      float(pers.std()), ent, float(b.mean()), float(dth.mean()),
                      *[float(((b <= r) & (dth > r)).sum()) for r in BETTI_RADII]])
    c = pts - pts.mean(0, keepdims=True)
    feats.extend([float(len(pts)), float(np.sqrt((c ** 2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(feats, dtype=np.float32)


def topo_for_chain(ca_xyz, per_res):
    atoms = np.concatenate([a for a in per_res if len(a)], axis=0)
    tree = cKDTree(atoms)
    out = np.zeros((len(ca_xyz), TOPO_DIM), np.float32)
    for i, ca in enumerate(ca_xyz):
        idx = tree.query_ball_point(ca, NBHD_RADIUS)
        if len(idx) > MAX_NBHD_ATOMS:
            _, idx = tree.query(ca, k=MAX_NBHD_ATOMS)
        pts = atoms[np.asarray(idx, dtype=int)]
        if len(pts) >= 4:
            out[i] = _vectorize(_diagrams(pts), pts)
    return out

## Cell 6: ANM (Anisotropic Network Model): compute low-frequency modes and generate perturbed conformations

In [ ]:
N_CONFORMERS = 8
TARGET_RMSD = 1.2   # Angstrom, typical moderate ANM thermal-fluctuation scale
N_MODES = 10
ANM_CUTOFF = 13.0


def anm_modes(ca, cutoff=ANM_CUTOFF, n_modes=N_MODES, gamma=1.0):
    """Standard ANM Hessian (Cα-only elastic network): eigenvectors give
    directional (x,y,z) per-residue displacement for each collective mode,
    unlike GNM's scalar/isotropic modes -- needed here because we want to
    actually DISPLACE atoms, not just estimate fluctuation magnitude."""
    n = len(ca)
    if n < 4:
        return np.zeros((0,)), np.zeros((0, n, 3))
    diff = ca[:, None, :] - ca[None, :, :]           # diff[i,j] = ca[i]-ca[j]
    dist2 = (diff ** 2).sum(-1)
    np.fill_diagonal(dist2, np.inf)
    contact = dist2 < cutoff ** 2
    outer = diff[:, :, :, None] * diff[:, :, None, :]
    blocks = np.where(contact[:, :, None, None], -gamma * outer / dist2[:, :, None, None], 0.0)
    diag = -blocks.sum(axis=1)
    idxr = np.arange(n)
    blocks[idxr, idxr] = diag
    H = blocks.transpose(0, 2, 1, 3).reshape(3 * n, 3 * n)
    eigvals, eigvecs = np.linalg.eigh(H)
    nz = np.where(eigvals > 1e-6)[0][:n_modes]
    if len(nz) == 0:
        return np.zeros((0,)), np.zeros((0, n, 3))
    modes = eigvecs[:, nz].T.reshape(len(nz), n, 3)
    return eigvals[nz], modes


def perturb_conformation(ca, modes, eigvals, target_rmsd, rng):
    if len(eigvals) == 0:
        return ca.copy()
    coeffs = rng.normal(0, 1, size=len(eigvals)) / np.sqrt(eigvals)
    disp = np.tensordot(coeffs, modes, axes=(0, 0))
    cur = np.sqrt((disp ** 2).sum(1).mean())
    if cur > 1e-8:
        disp = disp * (target_rmsd / cur)
    return ca + disp


def topo_variability_for_chain(ca_xyz, per_res, seed):
    """Returns (static_topo, variability_topo): static is byte-identical
    to Phase42's topo_for_chain output; variability is the per-residue
    STD of that same feature vector across the original + 8 ANM-
    perturbed conformations."""
    rng = np.random.default_rng(seed)
    static = topo_for_chain(ca_xyz, per_res)
    eigvals, modes = anm_modes(ca_xyz)
    ensemble = [static]
    for _ in range(N_CONFORMERS):
        ca_pert = perturb_conformation(ca_xyz, modes, eigvals, TARGET_RMSD, rng)
        disp = ca_pert - ca_xyz
        per_res_pert = [atoms + disp[i] for i, atoms in enumerate(per_res)]
        ensemble.append(topo_for_chain(ca_pert, per_res_pert))
    stacked = np.stack(ensemble, axis=0)
    var = stacked.std(axis=0)
    return static, var

## Cell 7: Compute static + variability topology for every chain (9 conformers each: 1 original + 8 ANM-perturbed)

In [ ]:
n_res_total = sum(len(r["ca"]) for r in records)
print(f"computing persistent homology on {n_res_total:,} residues x 9 conformers "
      f"({os.cpu_count()} workers, ~6.4 ms/residue/conformer -> ETA {n_res_total*0.0064*9/60:.0f} min)")
t0 = time.time()
results = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=5)(
    delayed(topo_variability_for_chain)(r["ca"], r["per_res"], seed=i) for i, r in enumerate(records))
el = time.time() - t0
print(f"\ndone in {el/60:.1f} min ({1000*el/max(n_res_total,1)/9:.2f} ms/residue/conformer)")

topo_all = [r[0] for r in results]
topovar_all = [r[1] for r in results]

topo_cat = np.concatenate(topo_all, axis=0)
keep = topo_cat.std(axis=0) > 1e-9
dropped = [TOPO_NAMES[i] for i in range(TOPO_DIM) if not keep[i]]
names = [TOPO_NAMES[i] for i in range(TOPO_DIM) if keep[i]]
print(f"static: pruned {len(dropped)} constant column(s): {dropped}")
mu, sd = topo_cat[:, keep].mean(0), topo_cat[:, keep].std(0); sd[sd < 1e-6] = 1.0

topovar_cat = np.concatenate(topovar_all, axis=0)
keep_var = topovar_cat.std(axis=0) > 1e-9
dropped_var = [TOPO_NAMES[i] for i in range(TOPO_DIM) if not keep_var[i]]
names_var = [f"var_{TOPO_NAMES[i]}" for i in range(TOPO_DIM) if keep_var[i]]
print(f"variability: pruned {len(dropped_var)} constant column(s): {dropped_var}")
mu_var, sd_var = topovar_cat[:, keep_var].mean(0), topovar_cat[:, keep_var].std(0); sd_var[sd_var < 1e-6] = 1.0

g = torch.Generator().manual_seed(0)
Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g))
Q = (Q * torch.sign(torch.det(Q))).numpy().astype(np.float32)
shift = np.array([31.7, -12.4, 88.1], dtype=np.float32)
ca0, pr0 = records[0]["ca"][:40], records[0]["per_res"][:40]
f_ref = topo_for_chain(ca0, pr0)
f_rot = topo_for_chain(ca0 @ Q.T + shift, [a @ Q.T + shift for a in pr0])
rel = float((np.abs(f_ref - f_rot) / np.maximum(np.abs(f_ref).max(0), 1e-6)).max())
print(f"static topology rotation invariance: {rel:.2e}")
assert rel < 1e-3

## Cell 8: Recompute backbone dihedrals (identical to Phase42)

In [ ]:
AA3 = set(AA3_TO_1)

def dihedrals_for(pid, path, cid, n_expect):
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[cid]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3 and "CA" in r]
    if len(residues) != n_expect:
        return None
    ppmap={}
    for pp in PPBuilder().build_peptides(ch, aa_only=False):
        for res,(phi,psi) in zip(pp, pp.get_phi_psi_list()):
            ppmap[res.id]=(phi,psi)
    phis,psis=[],[]
    for r in residues:
        phi,psi = ppmap.get(r.id, (None,None))
        phis.append(phi); psis.append(psi)
    def enc(angles):
        v=torch.tensor([a if a is not None else 0.0 for a in angles], dtype=torch.float32)
        m=torch.tensor([a is not None for a in angles], dtype=torch.float32)
        return torch.stack([torch.sin(v)*m, torch.cos(v)*m],dim=1), m.unsqueeze(1)
    pf,pm = enc(phis); sf,sm = enc(psis)
    return torch.cat([pf,sf,pm,sm],dim=1)

t0=time.time()
dihed = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(dihedrals_for)(r["pdb_id"], paths[r["pdb_id"]], r["chain_id"], len(r["ca"]))
    for r in records)
n_ok = sum(1 for x in dihed if x is not None)
print(f"\ndihedrals recovered for {n_ok:,}/{len(records):,} chains in {(time.time()-t0)/60:.1f} min")

## Cell 9: ESM2 embeddings (identical to Phase42)

In [ ]:
from transformers import AutoTokenizer, AutoModel
ESM_NAME = "facebook/esm2_t12_35M_UR50D"
tok = AutoTokenizer.from_pretrained(ESM_NAME)
esm = AutoModel.from_pretrained(ESM_NAME).to(DEVICE).eval()
ESM_DIM = esm.config.hidden_size
print(f"{ESM_NAME}: hidden {ESM_DIM}, device {DEVICE}")

@torch.no_grad()
def embed_batch(seqs, max_len=1502):
    enc = tok(seqs, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = esm(**enc).last_hidden_state
    embs = []
    for i, s in enumerate(seqs):
        n = len(s)
        embs.append(out[i, 1:1+n, :].float().cpu())
    return embs

order = sorted(range(len(records)), key=lambda i: len(records[i]["seq"]))
BATCH = 16
t0=time.time()
esm_feats = [None]*len(records)
for bi in range(0, len(order), BATCH):
    idx = order[bi:bi+BATCH]
    seqs = [records[i]["seq"] for i in idx]
    embs = embed_batch(seqs)
    for i, e in zip(idx, embs):
        esm_feats[i] = e
    if bi % (BATCH*20) == 0:
        print(f"  {bi}/{len(order)}  [{(time.time()-t0)/60:.1f}m]")

n_res = sum(len(r["ca"]) for r in records)
print(f"\nESM2 embeddings for {len(records):,} chains / {n_res:,} residues "
      f"in {(time.time()-t0)/60:.1f} min ({1000*(time.time()-t0)/max(n_res,1):.2f} ms/residue)")
del esm; torch.cuda.empty_cache()

## Cell 10: Assemble PyG graphs (identical to Phase42's schema, PLUS the new variability block appended)

In [ ]:
AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWYX"
AA_TO_IDX = {a: i for i, a in enumerate(AA_ALPHABET)}

def radius_graph_capped(pos, r, max_neighbors=32):
    n = pos.size(0)
    pos = pos - pos.mean(dim=0, keepdim=True)
    dist = torch.cdist(pos, pos).clone()
    dist.fill_diagonal_(float("inf"))
    mask = dist <= r
    if max_neighbors is not None and max_neighbors < n:
        _, knn = torch.topk(dist, max_neighbors, dim=1, largest=False)
        keep_ = torch.zeros_like(mask); keep_.scatter_(1, knn, True); mask &= keep_
    mask = mask | mask.t()
    tgt, src = mask.nonzero(as_tuple=True)
    return torch.stack([src, tgt], dim=0).long()

N_PRE = 21 + 6 + ESM_DIM
dataset = []
n_dropped = 0
for i, r in enumerate(records):
    if dihed[i] is None or esm_feats[i] is None:
        n_dropped += 1; continue
    n = len(r["ca"])
    if dihed[i].shape[0] != n or esm_feats[i].shape[0] != n:
        n_dropped += 1; continue
    onehot = torch.zeros(n, len(AA_ALPHABET))
    onehot[torch.arange(n), torch.tensor([AA_TO_IDX.get(c, 20) for c in r["seq"]])] = 1.0
    topo_i = topo_all[i][:, keep]
    topo_z = torch.from_numpy(((topo_i - mu) / sd).astype(np.float32))
    topovar_i = topovar_all[i][:, keep_var]
    topovar_z = torch.from_numpy(((topovar_i - mu_var) / sd_var).astype(np.float32))
    x = torch.cat([onehot, dihed[i], esm_feats[i], topo_z, topovar_z], dim=1)

    pos = torch.from_numpy(r["ca"]); pos = pos - pos.mean(0, keepdim=True)
    ei = radius_graph_capped(pos, 8.0, 32)
    d_ = (pos[ei[0]]-pos[ei[1]]).norm(dim=1,keepdim=True)
    sep = (ei[0]-ei[1]).abs().float().unsqueeze(1)

    y_np = r["y"].astype(bool)
    true_center = torch.from_numpy(r["ca"][y_np].mean(axis=0) - r["ca"].mean(axis=0)) if y_np.any() else None
    if true_center is None:
        n_dropped += 1; continue

    d = Data(x=x, pos=pos, edge_index=ei, edge_attr=torch.cat([d_, torch.log1p(sep)], 1),
             y=torch.from_numpy(r["y"]))
    d.true_center = true_center.float()
    d.pdb_id, d.chain_id, d.split = r["pdb_id"], r["chain_id"], r["split"]
    d.num_nodes = n
    assert torch.isfinite(d.x).all()
    dataset.append(d)

print(f"built {len(dataset):,} graphs ({n_dropped} dropped: alignment/labeling failures)")
n_res_ = sum(d.num_nodes for d in dataset); n_pos_ = sum(int(d.y.sum()) for d in dataset)
POS_WEIGHT = (n_res_ - n_pos_) / max(n_pos_, 1)
print(f"residues {n_res_:,} | cryptic positives {n_pos_:,} ({100*n_pos_/max(n_res_,1):.2f}%) "
      f"| pos_weight {POS_WEIGHT:.2f}")
print(f"static topo dim {int(keep.sum())} + variability topo dim {int(keep_var.sum())} "
      f"= total topo dim {int(keep.sum())+int(keep_var.sum())}")
for s in ("train","test"):
    ds = [d for d in dataset if d.split == s]
    print(f"  {s:<6} {len(ds):>5} chains")

## Cell 11: Carve a val subset from train, save (identical convention to Phase42, plus variability metadata)

In [ ]:
rng = np.random.RandomState(42)
train_idx = [i for i,d in enumerate(dataset) if d.split == "train"]
perm = rng.permutation(len(train_idx))
n_val = max(1, int(0.08 * len(perm)))
val_set = {train_idx[i] for i in perm[:n_val]}
for i, d in enumerate(dataset):
    if i in val_set: d.split = "val"

for s in ("train","val","test"):
    ds = [d for d in dataset if d.split == s]
    r_ = sum(d.num_nodes for d in ds); p_ = sum(int(d.y.sum()) for d in ds)
    print(f"{s:<6} chains {len(ds):>5}  residues {r_:>7,}  positives {p_:>5}  rate {100*p_/max(r_,1):>5.2f}%")

for d in dataset:
    assert torch.isfinite(d.x).all() and torch.isfinite(d.pos).all()
    assert d.edge_index.max() < d.num_nodes and d.edge_index.min() >= 0
print("integrity OK")

STATIC_TOPO_DIM = int(keep.sum())
VAR_TOPO_DIM = int(keep_var.sum())
p = WORK / "phase157_cryptobench_topovar_graphs.pt"
torch.save({"dataset": dataset, "n_pre_topo": N_PRE,
            "topo_dim": STATIC_TOPO_DIM + VAR_TOPO_DIM,
            "static_topo_dim": STATIC_TOPO_DIM, "var_topo_dim": VAR_TOPO_DIM,
            "topo_names": names + names_var, "burial_names": BURIAL_NAMES, "dropped_names": dropped + dropped_var,
            "topo_mean": mu, "topo_std": sd, "topovar_mean": mu_var, "topovar_std": sd_var,
            "esm_dim": ESM_DIM, "pos_weight": POS_WEIGHT,
            "n_conformers": N_CONFORMERS, "target_rmsd": TARGET_RMSD, "anm_cutoff": ANM_CUTOFF,
            "aa_alphabet": AA_ALPHABET, "source": "CryptoBench (Skrhak et al. 2025, btae745)"}, p)
print(f"saved -> {p} ({p.stat().st_size/1e6:.0f} MB)")